In [5]:
import os
import numpy as np
import pandas as pd
import librosa
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
)
import joblib

In [6]:
path = "/home/habib/mindcloud/project/features_dtst.csv"
dtst = pd.read_csv(path)
path = "/home/habib/mindcloud/project/dataset.csv"
dataset = pd.read_csv(path)
# Drops rows where Cry_Reason is 0 or 5, and updates the DataFrame
dtst = dtst[~dtst["Cry_Reason"].isin([0, 1])]

In [7]:
Cry_Audio_Filel=[]
Cry_Reasonl = []
MFCCs13Meanl = []
ZCR_Meanl = []
RMS_Meanl = []
duration = []
F0l = []

for i,row1 in dtst.iterrows():
    for j, row2 in dataset.iterrows():
        if os.path.basename(row1["Cry_Audio_File"]) == os.path.basename(row2["x"]):
            Cry_Audio_Filel.append(row2["x"])
            Cry_Reasonl.append(row1["Cry_Reason"])
            MFCCs13Meanl.append(row1["MFCCs13Mean"])
            ZCR_Meanl.append(row1["ZCR_Mean"])
            RMS_Meanl.append(row1["RMS_Mean"])


In [8]:
target_count = (dtst["Cry_Reason"] == 3).sum()
print(f"hungry = {target_count}")
discomfort_count = (dtst["Cry_Reason"] == 2).sum()
tired_count = (dtst["Cry_Reason"] == 4).sum()
print(f"discomfort = {discomfort_count}")
print(f"tired = {tired_count}")

hungry = 382
discomfort = 27
tired = 24


In [9]:
csv_output_path = "/home/habib/mindcloud/project/features_dtst.csv"
dtst.to_csv(csv_output_path, index=False)

In [10]:
dt = pd.read_csv(csv_output_path)

In [11]:
def apply_pitch_shift(y, sr):
    # Choose a random semitone shift between -2 and +2
    n_steps = np.random.uniform(-2.0, 2.0)
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def apply_time_stretch(y):
    # Choose a random speed factor between 0.9 and 1.1
    rate = np.random.uniform(0.9, 1.1)
    return librosa.effects.time_stretch(y, rate=rate)

def apply_add_noise(y):
    # Generate random noise matching the audio length
    noise = np.random.normal(0, y.std(), len(y))
    # Mix it in lightly (0.005 is a safe factor for light noise)
    noise_factor = 0.005 
    return y + noise_factor * noise

In [12]:
output_dir = "/home/habib/mindcloud/project/dataset/augmented/"
os.makedirs(output_dir, exist_ok=True)



In [13]:
X = dt[["MFCCs13Mean"]].values
y = dt["Cry_Reason"].values

# 2. Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [14]:
# -------------------------------------------------------------------------
# 3. COST-SENSITIVE MACHINE LEARNING SETUP
# -------------------------------------------------------------------------
print("\nInitializing Cost-Sensitive Random Forest...")

# 'balanced_subsample' computes cost weights dynamically per class 
# for every single split branch of every single tree in the forest.
# It places extreme penalties on misclassifying minority items.
model = RandomForestClassifier(
    n_estimators=200,                  # High estimator count to stabilize internal cost penalties
    criterion="gini",
    max_depth=10,                      # Caps tree splits to force generalization over features
    min_samples_split=5,               # Prevents splitting on unique noise signatures
    class_weight="balanced_subsample", # Core cost-sensitive penalty engine
    random_state=42,
    n_jobs=-1                          # Uses all laptop CPU threads for faster execution
)

# Train the estimator strictly on the imbalanced raw data matrix
model.fit(X_train, y_train)

# -------------------------------------------------------------------------
# 4. EVALUATE PIPELINE RESULTS
# -------------------------------------------------------------------------
# Predict on test data
y_pred = model.predict(X_test)

# 1. Calculate Standard Accuracy
acc = accuracy_score(y_test, y_pred)
print(f"\n Standard Accuracy: {acc * 100:.2f}%")

# 2. Calculate Balanced Accuracy (Highly recommended for imbalanced data!)
balanced_acc = balanced_accuracy_score(y_test, y_pred)
print(f" Balanced Accuracy: {balanced_acc * 100:.2f}%")

print("\n--- Cost-Sensitive Classification Report ---")
print(classification_report(y_test, y_pred))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))



Initializing Cost-Sensitive Random Forest...

 Standard Accuracy: 75.86%
 Balanced Accuracy: 28.57%

--- Cost-Sensitive Classification Report ---
              precision    recall  f1-score   support

           2       0.00      0.00      0.00         5
           3       0.87      0.86      0.86        77
           4       0.00      0.00      0.00         5

    accuracy                           0.76        87
   macro avg       0.29      0.29      0.29        87
weighted avg       0.77      0.76      0.76        87

--- Confusion Matrix ---
[[ 0  5  0]
 [ 8 66  3]
 [ 0  5  0]]


In [15]:
# Save the trained model cleanly to your project workspace
model_filename = "/home/habib/mindcloud/project/cost_sensitive_rf_model2.joblib"
joblib.dump(model, model_filename)
print(f"\nModel successfully saved to {model_filename}")


Model successfully saved to /home/habib/mindcloud/project/cost_sensitive_rf_model2.joblib
